# Tutorial 3 — Operators and Gradients

The same formula can be evaluated with different fuzzy operators. They can give similar truth values and still **train very differently**, because what matters for learning is the gradient each operator sends to the predicates and the other learnable symbols.

This tutorial:

1. compares four implications on the same formula, and shows which individuals receive a gradient under each;
2. shows how the `p` of the universal quantifier decides **where** the gradient goes;
3. lets you explore every connective as a surface over $[0,1]^2$, its value or the size of its gradient, with sliders for $p$ (the exponent of the p-means: larger is closer to max or min), $\varepsilon$ (how far `stable` moves inputs away from 0 and 1) and $\beta$ (the sharpness of the smoothed operators: larger is closer to the hard version);
4. shows `smooth_clamp`, which some operators use to stay differentiable.

> **Extra dependencies for sections 3–4:** `matplotlib`, `ipywidgets` (`pip install sltn[notebooks]`).

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Interpretation, Logic
from sltn.interpretation.operators import aggregator, implication

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setup

The animals of Tutorials 1 and 2: an animal is a vector in ℝ² (size, furriness). This time both predicates are **trainable networks**, so that we can look at the gradients they receive.

`Net` keeps its last output and asks PyTorch to keep that output's gradient (`retain_grad`). After a backward pass, `net.out.grad` holds $\partial\,\mathrm{sat} / \partial\, \mathrm{IsDog}(x_i)$: how much the satisfaction would change if the network's answer for animal $i$ changed. This is only for inspection; a normal network doesn't need it.

`make_interp(logic)` builds an interpretation with the given `Logic` (`None` for the default) and the **same** networks and data, so the only difference between experiments is the operators.

In [ ]:
sig = Signature("animals")
sig.sort("Animal")
sig.variable("x", "Animal")
sig.predicate("IsDog", ["Animal"])
sig.predicate("IsLarge", ["Animal"])


class Net(nn.Module):
    """A small predicate network that keeps its last output, to read its gradient."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())

    def forward(self, a):                 # a: [B, 2]
        self.out = self.net(a)            # [B, 1]
        self.out.retain_grad()
        return self.out


is_dog, is_large = Net(), Net()
animals = torch.randn(16, 2, device=device)


def make_interp(logic=None):
    interp = Interpretation(sig, logic=logic).to(device)
    interp["Animal"] = sltn.Type("R2", shape=(2,))
    interp["x"] = animals
    interp["IsDog"] = is_dog
    interp["IsLarge"] = is_large
    return interp


def output_grad(net):
    """The gradient of the last backward pass w.r.t. the network's output, one value per animal."""
    return net.out.grad[:, 0] if net.out.grad is not None else torch.zeros(len(animals), device=device)

## 1 — Four implications, four learning signals

The formula is "every dog is large", `∀ x: (IsDog(x) → IsLarge(x))`. With $a = \mathrm{IsDog}(x)$ and $b = \mathrm{IsLarge}(x)$:

| Implication | $a \to b$ | Where it is 1 |
|---|---|---|
| **Reichenbach** (the default, as in LTN) | $1 - a + a b$ | only at $a = 0$ or $b = 1$ |
| **Goguen** | 1 if $a \le b$, else $b / a$ | $a \le b$ |
| **Łukasiewicz** | $\min(1,\ 1 - a + b)$ | $a \le b$ |
| **Gödel** | 1 if $a \le b$, else $b$ | $a \le b$ |

The `Logic` is passed to the interpretation with `logic=`. Only the implication changes; `∀` stays pMeanError with p = 2.

In [ ]:
phi = sig.parse("forall x: (IsDog(x) -> IsLarge(x))")

configs = {
    "Reichenbach (default)": None,
    "Goguen":                Logic(Implies=implication.ImpliesGoguen()),
    "Łukasiewicz":           Logic(Implies=implication.ImpliesLuk()),
    "Gödel":                 Logic(Implies=implication.ImpliesGodel()),
}

grads = {}
print(f"{'implication':<24s} {'sat':>6s}   no gradient   mean |∂sat/∂IsLarge|: a ≤ b    a > b")
for name, logic in configs.items():
    interp = make_interp(logic)
    sat = interp(phi).value
    sat.backward()

    g_dog, g_large = output_grad(is_dog), output_grad(is_large)
    grads[name] = g_large.clone()
    holds = (is_dog.out <= is_large.out)[:, 0]               # animals where a <= b
    no_grad = ((g_dog.abs() < 1e-6) & (g_large.abs() < 1e-6)).sum().item()
    print(f"{name:<24s} {sat.item():6.3f}   {no_grad:2d} of {len(animals)}"
          f"      {g_large[holds].abs().mean().item():22.4f}  {g_large[~holds].abs().mean().item():8.4f}")
    interp.zero_grad()

print(f"\n{holds.sum().item()} of {len(animals)} animals have IsDog(x) <= IsLarge(x)")

What to read in the table:

- **Goguen and Gödel give no gradient at all to the animals where $a \le b$.** For them the implication is already exactly 1, and it stays 1 if $a$ or $b$ move a little, so the networks learn nothing from those animals. The other animals carry all the signal.
- **Reichenbach gives a gradient to every animal.** $1 - a + ab$ is 1 only at $a = 0$ or $b = 1$, so it keeps pushing $a$ down and $b$ up even where $a \le b$. This is the default, as in LTN: a denser signal, but it also pushes animals that already satisfy the rule.
- **Łukasiewicz** gives a small gradient where $a \le b$ only because it is smoothed: operators are stable by default, and for Łukasiewicz `stable` replaces the hard $\min(1, \cdot)$ with a smooth one. With `ImpliesLuk(stable=False)` it behaves like Goguen there.
- The truth values differ too, but less than the gradients. **Gödel** also sends nothing to `IsDog`: where $a > b$ its value is $b$, which doesn't depend on $a$.

The gradient each animal sends to `IsLarge`, for the first animals, under Reichenbach and Goguen:

In [ ]:
a, b = is_dog.out.detach()[:, 0], is_large.out.detach()[:, 0]
print("animal   IsDog  IsLarge   a <= b    ∂sat/∂IsLarge: Reichenbach   Goguen")
for i in range(6):
    print(f"{i:>6d}   {a[i]:.3f}    {b[i]:.3f}   {str(bool(a[i] <= b[i])):>6s}"
          f"   {grads['Reichenbach (default)'][i].item():26.4f}   {grads['Goguen'][i].item():8.4f}")

## 2 — Where the gradient goes: the `p` of ∀

`∀` is pMeanError, $1 - \big(\mathrm{mean}_i\, (1 - a_i)^p\big)^{1/p}$. Its gradient for individual $i$ is proportional to $(1 - a_i)^{p-1}$: the less an individual satisfies the formula, the more of the gradient it gets, and `p` decides **how much more**.

- $p = 1$ is the plain mean: every individual gets the same share of the gradient.
- A larger $p$ moves the value towards the minimum, and the gradient towards the worst individuals. As $p \to \infty$, only the worst one counts.

Below, for `∀ x: IsDog(x)`, the value and the share of the gradient that goes to the 4 least dog-like of the 16 animals. With p = 1 that share is exactly 4/16 = 25%.

In [ ]:
interp = make_interp()
psi = sig.parse("forall x: IsDog(x)")

for p in [1, 2, 4, 10]:
    interp.logic["∀"] = aggregator.AggregPMeanError(p=p)
    sat = interp(psi).value
    sat.backward()

    g = output_grad(is_dog)
    worst = is_dog.out.detach()[:, 0].argsort()[:4]          # the 4 least dog-like animals
    print(f"p={p:2d}   ∀ x: IsDog(x) = {sat.item():.4f}   gradient share of the 4 worst: {(g[worst].sum() / g.sum()).item():.0%}")
    interp.zero_grad()

p = 2 (the default) is a middle ground: the worst individuals count more, but the others still learn. A large p focuses on outliers, which helps to enforce a rule strictly but makes training noisier.

## 3 — Connectives as surfaces

The cell below plots a connective over the unit square $[0,1]^2$, with `Input 1` as its first argument and `Input 2` as its second. **Show** selects what is plotted:

- **value** — the truth value, from 0 (dark) to 1 (light), with a black contour line every 0.05;
- **|gradient|** — the size of its gradient, $\log_{10} \lVert \nabla f(x, y) \rVert$. The darkest regions (below $10^{-3}$) are **flat**: an input there gets almost no learning signal. The lightest regions are steep: a small change gives a large gradient. The contour lines are every half decade (a factor of about 3).

What to look for:

- **Flat regions:** Goguen, Gödel and Łukasiewicz implications are flat above the diagonal ($x \le y$), as in section 1; `And (Min)` sends gradient to only one input at a time.
- **Steep regions:** Goguen is steep near $x = 0$ below the diagonal, where $y / x$ changes fast.
- **p:** for the p-means, a larger $p$ sharpens the surface towards max (`p-Mean`) or min (`p-Mean Error`).
- **ε:** operators are stable, so inputs are moved $\varepsilon$ away from 0 and 1. A large $\varepsilon$ visibly changes the values; the default is $10^{-6}$.
- **β:** for the smoothed operators (Łukasiewicz, smooth Gödel and Goguen), a larger $\beta$ is closer to the hard operator.

The two p-means are aggregators, here applied to two values, `torch.stack([x, y])`, as a quantifier would aggregate two individuals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, FloatLogSlider
import sltn.interpretation.operators as ops

In [ ]:
def make_operator(name, p, eps, beta):
    """The selected connective, as a function of two tensors."""
    pair = lambda agg: (lambda x, y: agg(torch.stack(torch.broadcast_tensors(x, y)), dim=[0]))
    return {
        "And (Prod)":               ops.AndProd(eps=eps),
        "And (Min)":                ops.AndMin(),
        "And (Łukasiewicz)":        ops.AndLuk(eps=eps, beta=beta),
        "And (pMean)":              ops.AndPMean(p=p, eps=eps),
        "Or (ProbSum)":             ops.OrProbSum(eps=eps),
        "Or (Max)":                 ops.OrMax(),
        "Implies (Goguen)":         ops.ImpliesGoguen(eps=eps),
        "Implies (Goguen, smooth)": ops.ImpliesGoguen(eps=eps, smooth=True, beta=beta),
        "Implies (Reichenbach)":    ops.ImpliesReichenbach(eps=eps),
        "Implies (Łukasiewicz)":    ops.ImpliesLuk(eps=eps, beta=beta),
        "Implies (Gödel)":          ops.ImpliesGodel(),
        "Implies (Gödel, smooth)":  ops.ImpliesGodel(smooth=True, beta=beta),
        "Implies (Kleene-Dienes)":  ops.ImpliesKleeneDienes(),
        "Equiv (Prod + Reichenbach)": ops.Equiv(and_op=ops.AndProd(eps=eps), implies_op=ops.ImpliesReichenbach(eps=eps)),
        "Equiv (Łukasiewicz)":      ops.Equiv(and_op=ops.AndLuk(eps=eps, beta=beta), implies_op=ops.ImpliesLuk(eps=eps, beta=beta)),
        "Equiv (Similarity)":       ops.EquivSimilarity(p=p, eps=eps),
        "p-Mean (∃)":               pair(ops.AggregPMean(p=p, eps=eps)),
        "p-Mean Error (∀)":         pair(ops.AggregPMeanError(p=p, eps=eps)),
    }[name]


def plot_connective(name, show, p, eps, beta):
    grid = torch.linspace(0, 1, 101)
    X, Y = torch.meshgrid(grid, grid, indexing="xy")        # X: Input 1 (columns), Y: Input 2 (rows)
    X.requires_grad_(True)
    Y.requires_grad_(True)
    Z = make_operator(name, p, eps, beta)(X, Y)

    if show == "value":
        data = Z.detach()
        levels, lines, ticks, label = np.linspace(0, 1, 101), np.linspace(0, 1, 21), np.linspace(0, 1, 6), "truth value"
    else:
        gx, gy = torch.autograd.grad(Z.sum(), [X, Y], allow_unused=True)
        gx = torch.zeros_like(X) if gx is None else gx
        gy = torch.zeros_like(Y) if gy is None else gy
        data = torch.log10(torch.sqrt(gx ** 2 + gy ** 2).clamp_min(1e-3)).clamp_max(2)
        levels, lines, ticks, label = np.linspace(-3, 2, 101), np.linspace(-3, 2, 11), np.arange(-3, 3), "log10 |gradient|"

    fig, ax = plt.subplots(figsize=(6, 5))
    contour = ax.contourf(grid.numpy(), grid.numpy(), data.numpy(), levels=levels, cmap="viridis")
    ax.contour(grid.numpy(), grid.numpy(), data.numpy(), levels=lines, colors="k", linewidths=0.5)
    fig.colorbar(contour, ax=ax, ticks=ticks, label=label)
    ax.set_title(f"{name}   (p={p:g}, ε={eps:.0e}, β={beta:g})")
    ax.set_xlabel("Input 1 (x)")
    ax.set_ylabel("Input 2 (y)")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


interact(
    plot_connective,
    name=Dropdown(options=[
        "And (Prod)", "And (Min)", "And (Łukasiewicz)", "And (pMean)",
        "Or (ProbSum)", "Or (Max)",
        "Implies (Goguen)", "Implies (Goguen, smooth)", "Implies (Reichenbach)", "Implies (Łukasiewicz)",
        "Implies (Gödel)", "Implies (Gödel, smooth)", "Implies (Kleene-Dienes)",
        "Equiv (Prod + Reichenbach)", "Equiv (Łukasiewicz)", "Equiv (Similarity)",
        "p-Mean (∃)", "p-Mean Error (∀)",
    ], value="Implies (Reichenbach)", description="Connective:"),
    show=Dropdown(options=["value", "|gradient|"], value="|gradient|", description="Show:"),
    p=FloatSlider(min=0.5, max=10.0, step=0.5, value=2.0, description="p:", continuous_update=False),
    eps=FloatLogSlider(base=10, min=-8, max=-1, step=0.5, value=1e-6, description="ε:", continuous_update=False),
    beta=FloatSlider(min=0.5, max=20.0, step=0.5, value=10.0, description="β:", continuous_update=False),
);

**What `stable` does, and what it doesn't.** With `stable=True` (the default for every operator), the inputs are moved $\varepsilon$ away from 0 and 1 (`pi_0`, `pi_1` in `sltn.interpretation.operators.utils`) before the operator is applied. This prevents **infinite or NaN gradients at exactly 0 and 1**, e.g. pMeanError when every individual is fully true, or Goguen at $x = 0$. It does **not** remove flat regions: Goguen above the diagonal or `And (Min)` stay flat, because that's what the operators are. To get a gradient there, choose another operator (section 1) or a smooth version.

## 4 — Smooth clamp

Several operators (Łukasiewicz, and Gödel and Goguen with `smooth=True`) replace a hard `torch.clamp(x, 0, 1)` with `smooth_clamp(x, beta)`, which is differentiable everywhere. The plot compares the two. A larger $\beta$ is closer to the hard clamp, and less smooth. The Łukasiewicz operators use `squareplus`, the smooth Gödel and Goguen use `softplus`.

In [ ]:
from sltn.interpretation.operators.utils import smooth_clamp

x_clamp = torch.linspace(-1.5, 2.5, 800)


def plot_smooth_clamp(beta, method):
    y_smooth = smooth_clamp(x_clamp, min=0., max=1., beta=beta, smoothing_method=method)
    y_hard = torch.clamp(x_clamp, 0., 1.)

    plt.figure(figsize=(8, 4.5))
    plt.plot(x_clamp.numpy(), y_smooth.numpy(), label=f"smooth_clamp ({method}, β={beta:g})", linewidth=2)
    plt.plot(x_clamp.numpy(), y_hard.numpy(), "--", label="hard clamp [0, 1]", linewidth=2)
    plt.ylim(-0.05, 1.05)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Smooth clamp vs hard clamp")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


interact(
    plot_smooth_clamp,
    beta=FloatSlider(min=0.5, max=20.0, step=0.5, value=10.0, description="β:", continuous_update=False),
    method=Dropdown(options=["squareplus", "softplus"], value="squareplus", description="method:"),
);

---

**Next:** [Tutorial 4 — KB & Learning](4-KB&Learning.ipynb) puts everything together in an end-to-end training loop over a knowledge base of formulas.